In [0]:
files = {
    "employee" : "/Volumes/workspace/default/assignment/employees_extended.json",
    "Transaction" : "/Volumes/workspace/default/assignment/transactions_1200 (1).json"
}
df = {}

for name, path in files.items():
    df[name]=spark.read.json(
        path
    )

In [0]:
df["employee"].display()

address,age,department,emp_id,is_active,join_date,metadata,name,projects,salary,skills
"List(Hyderabad, TS)",26,Operations,1,false,2024-02-12,"List(L3, null, manual)",Ravi K,"List(List(35, Alpha), List(131, Alpha), List(278, Beta))",59378.66,List(GCP)
"List(Bangalore, KA)",30,Sales,2,true,2021-10-25,"List(L2, null, ingest)",Anu S,List(),null,"List(Spark, Scala, Excel)"
"List(Hyderabad, TS)",55,IT,3,true,2025-01-18,"List(null, null, api)",Anu V,List(),146639.47,"List(GCP, Scala, SQL)"
"List(Delhi, DL)",42,Operations,4,true,2020-12-29,"List(L3, null, api)",Kiran N,"List(List(393, Epsilon))",null,List(GCP)
"List(Hyderabad, TS)",22,Support,5,true,2022-07-02,"List(null, IN, null)",Divya B,"List(List(355, Beta), List(222, Delta))",121999.99,"List(SQL, Scala, GCP)"
"List(Hyderabad, TS)",57,Sales,6,false,2020-06-17,"List(null, IN, null)",Vikram V,List(),118376.73,"List(Azure, AWS)"
"List(Kochi, KL)",36,Finance,7,false,2018-02-17,"List(L3, null, ingest)",Ravi J,"List(List(348, Omega), List(77, Gamma))",99486.05,List(Tableau)
"List(Kochi, KL)",39,Marketing,8,false,2023-09-11,"List(null, null, null)",Karthik M,"List(List(418, Epsilon))",139078.48,"List(Spark, Excel)"
"List(Chennai, TN)",39,Sales,9,false,2020-09-13,"List(null, EU, api)",Ravi C,"List(List(55, Omega), List(292, Omega), List(84, Omega))",134840.23,List(SQL)
"List(Bangalore, KA)",39,Support,10,true,2025-07-14,"List(L3, null, ingest)",Manoj V,List(),116241.38,"List(AWS, Azure, Airflow)"


In [0]:
df["Transaction"].display()

amount,emp_id,location,payment_mode,txn_date,txn_id
29350.8,1442,"List(Chennai, 584669)",Cash,2026-04-01,1
13724.09,1367,"List(Chennai, 685497)",Cash,2026-01-20,2
13781.27,1397,"List(Kochi, 567976)",UPI,2026-12-09,3
8771.17,1472,"List(Pune, 587233)",Wallet,2026-08-16,4
7744.27,1029,"List(Chennai, 594510)",Card,2024-10-23,5
31910.16,1074,"List(Hyderabad, 642375)",UPI,2024-01-15,6
30738.79,1089,"List(Pune, 647206)",UPI,2025-04-21,7
36134.56,1678,"List(Chennai, 695355)",Cash,2026-03-19,8
33419.35,1004,"List(Delhi, 577027)",UPI,2025-07-27,9
4290.04,1187,"List(Pune, 571667)",NetBanking,2024-12-09,10


### Q1: Array Explosion 

In [0]:
from pyspark.sql.functions import explode

df["employee"].select("emp_id","name",explode("skills").alias("skill")).display()

emp_id,name,skill
1,Ravi K,GCP
2,Anu S,Spark
2,Anu S,Scala
2,Anu S,Excel
3,Anu V,GCP
3,Anu V,Scala
3,Anu V,SQL
4,Kiran N,GCP
5,Divya B,SQL
5,Divya B,Scala


### Q2: Filter on Nested Field 

In [0]:
from pyspark.sql.functions import col
df["employee"].filter(col("address.city") == "Chennai").select("emp_id","name","address.city").display()

emp_id,name,city
9,Ravi C,Chennai
11,Ravi H,Chennai
16,Anu S,Chennai
40,Arun R,Chennai
46,Sana T,Chennai
51,Karthik T,Chennai
57,Rahul A,Chennai
60,Priya L,Chennai
62,Anu B,Chennai
64,Ravi B,Chennai


### Q3: Map Access

In [0]:
df["employee"].select("emp_id","name",
                      col("metadata")["level"].alias("level")).display()

emp_id,name,level
1,Ravi K,L3
2,Anu S,L2
3,Anu V,null
4,Kiran N,L3
5,Divya B,null
6,Vikram V,null
7,Ravi J,L3
8,Karthik M,null
9,Ravi C,null
10,Manoj V,L3


### Q4: Project Analysis 

In [0]:
from pyspark.sql.functions import sum, explode,count

df["employee"].select(
    "emp_id","name",explode("projects").alias("project")
    ).groupBy(
        "emp_id","name"
        ).agg(
            sum("project.hours").alias("Total_hours"),count(
                "project.project_name").alias("project_Count")
            ).display()

emp_id,name,Total_hours,project_Count
24,Vikram B,1234,3
25,Karthik A,376,1
40,Arun R,669,2
57,Rahul A,494,3
89,Sana H,89,1
108,Meena H,650,3
170,Anu K,562,2
175,Anu R,102,1
197,Meena A,1097,3
278,Manoj V,161,1


### Q5: High Skill Employees

In [0]:
df["employee"].select("emp_id","name", explode("skills").alias("skill")).groupBy("emp_id","name").agg(count("skill").alias("Total_skills")).filter(col("Total_skills") > 3).display()

emp_id,name,Total_skills
24,Vikram B,4
25,Karthik A,4
40,Arun R,4
46,Sana T,4
89,Sana H,4
315,Meena N,4
377,John C,4
430,Neha L,4
526,Karthik M,4
563,Arun H,4
